In [22]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-mpnet-base-v2")

# topic_text = """
# war, army, military, conflict, attack, bomb, invasion, battle, soldiers, violence,
# occupation, missile, fighting, defense, siege, casualties, combat, troops, conflict zones
# """
# topic_text = "government, parliament, prime minister, president, elections, policy, law, minister, democracy, legislation"
# topic_text = "economy, inflation, stock, market, business, trade, unemployment, budget, finance, investment, recession"
# topic_text = "protest, health, education, rights, inequality, social, law, welfare, healthcare, crime"
topic_text = "israel, palestine"
# topic_text = "russia, ukraine"

topic_emb = model.encode(topic_text, convert_to_tensor=True)

chunk = "By contrast, the bottom tier of technocrats appears relatively homogeneous – at least in terms of Palestinian identity – though it is reasonable to assume that both the Palestinian Authority and Hamas will exert pressure on each of its members to adopt positions closer to their own. The most difficult task is the demilitarization of Gaza and the disarmament of Hamas. At the Davos meeting, Jared Kushner presented a master plan for dismantling Hamas, demilitarizing the Strip, and rebuilding it, but it remains unclear who would carry out the first two tasks. The “international stabilization force” mentioned in Trump’s plan already has an American commander, but it has no army. Countries that committed to sending forces – such as Azerbaijan and Indonesia – have meanwhile backed out."
chunk_emb = model.encode(chunk, convert_to_tensor=True)

sim = util.cos_sim(chunk_emb, topic_emb)
print(sim)

tensor([[0.3524]])


In [ ]:
import pandas as pd
import re

class Months:
    MONTHS_WORDS1 = {"January": "01", "February": "02", "March": "03", "April": "04", "May": "05", "June": "06", "July": "07", "August": "08", "September": "09", "October": "10", "November": "11", "December": "12"}
    MONTHS_WORDS_ABBR = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06", "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
    MONTHS_RUSSIAN = {"января": "01", "февраля": "02", "марта": "03", "апреля": "04","мая": "05", "июня": "06", "июля": "07", "августа": "08","сентября": "09", "октября": "10", "ноября": "11", "декабря": "12"}
    DAYS_TO_REPLACE = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def clean_date():
    website = "jpost"
    df = pd.read_csv("../data/preprocessing/pal_isr/jpost_original.csv")
    
    pattern_en = r"\b(" + "|".join(Months.MONTHS_WORDS1.keys()) + r")\b"
    pattern_ru = r"\b(" + "|".join(Months.MONTHS_RUSSIAN.keys()) + r")\b"
    pattern_days = r"\b(" + "|".join(Months.DAYS_TO_REPLACE) + r")\b"
    pattern_abbr = r"\b(" + "|".join(Months.MONTHS_WORDS_ABBR.keys()) + r")\b"

    if website != "liganet":
        if website not in ["kpru", "alquds", "ukpravda"]:
            df["date"] = (df["date"].str.replace(r'(?<=\w) (?=\w)', '', regex=True))
        df["date"] = (df["date"]
            .str.replace(pattern_days, "", regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_en, lambda m: Months.MONTHS_WORDS1[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_ru, lambda m: Months.MONTHS_RUSSIAN[m.group(0).lower()], regex=True, flags=re.IGNORECASE)
            .str.replace(r"[,/]", " ", regex=True)
            .str.replace(r"\s+\d{1,2}\s*:\s*\d{2}.*$", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(pattern_abbr, lambda m: Months.MONTHS_WORDS_ABBR[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.strip())
        if website == "kpru":
            df["date"] = (df["date"].str.replace(" ", "-"))
            df["date"] = df["date"].apply(lambda x: x + "-2026" if isinstance(x, str) and x.count("-") == 1 else x)

        if website != "kpru":
            df["date"] = df["date"].str.replace(" ", "-")
            df["date"] = df["date"].apply(lambda x: x + "-2026" if isinstance(x, str) and x.count("-") == 1 else x)
            df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True).dt.strftime("%d-%m-%Y")
    
    print(df["date"].dtype)
    print(df["date"].tail(20))

clean_date()

object
216    NaN
217    NaN
218    NaN
219    NaN
220    NaN
221    NaN
222    NaN
223    NaN
224    NaN
225    NaN
226    NaN
227    NaN
228    NaN
229    NaN
230    NaN
231    NaN
232    NaN
233    NaN
234    NaN
235    NaN
Name: date, dtype: object


In [68]:
import matplotlib.pyplot as plt
website = "liganet"
df = pd.read_csv(f"../data/ukraine/{website}_embedded.csv")

filtered_df = df[(df["filter1"] > 0.4) | (df["filter2"] > 0.4) | (df["filter3"] > 0.4)].copy()
filtered_df.to_csv(f"../data/ukraine/{website}_filtered.csv")
print((filtered_df["title"].count() / df["title"].count())*100)

44.565217391304344


In [10]:
from transformers import pipeline

EMOTION_CLASSIFIER = pipeline(task = "text-classification", model = "j-hartmann/emotion-english-distilroberta-base", return_all_scores = True)
EMOTION_CLASSIFIER("But there is no doubt that the soldiers themselves will delay this moment to the last moment, knowing perfectly well that the weapons in Ukraine are fucked up and larger, and for any shot fired at the population, they will get at least five shots in their direction. It is not worth overestimating this phenomenon, considering it to be to some extent an expression of some ""prorussicity"" of Ukrainians. Until then, like ""before Kiev,"" and at least there is no ""pro-Russianity."" This is just an instinct for the self-preservation of civilians, albeit at a collective, higher level than that of singles. It is precisely for the system struggle against the regime of the Zelen and bandier power of the current ""cuffing"" of the population of the TCDs and the police that is clearly not enough.")

Device set to use cpu
c:\Users\sangi\Documents\GitHub\master-thesis\.venv\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': 'anger', 'score': 0.13161607086658478},
  {'label': 'disgust', 'score': 0.1095079556107521},
  {'label': 'fear', 'score': 0.006551758851855993},
  {'label': 'joy', 'score': 0.0023150774650275707},
  {'label': 'neutral', 'score': 0.7373749017715454},
  {'label': 'sadness', 'score': 0.00576796242967248},
  {'label': 'surprise', 'score': 0.0068662893027067184}]]

In [9]:
def clean_date(website):
        website = website
        df = pd.read_csv("../data/preprocessing/pal_isr/jpost_original.csv")
        print("length: ", df.count())
        # 1. Handle JPost "Exploded" format: "M A R C H   9 ,   2 0 2 6"
        # This regex removes single spaces that are between two alphanumeric characters
        # "M A R C H" -> "MARCH", "2 0 2 6" -> "2026"
        # It leaves multiple spaces (like between MARCH and 9) alone.
        if website == "jpost":
            df["date"] = df["date"].astype(str).str.replace(r'(?<=\w)\s(?=\w)', '', regex=True)

        # 2. Define standard patterns
        pattern_en = r"\b(" + "|".join(Months.MONTHS_WORDS1.keys()) + r")\b"
        pattern_ru = r"\b(" + "|".join(Months.MONTHS_RUSSIAN.keys()) + r")\b"
        pattern_days = r"\b(" + "|".join(Months.DAYS_TO_REPLACE) + r")\b"
        pattern_abbr = r"\b(" + "|".join(Months.MONTHS_WORDS_ABBR.keys()) + r")\b"

        # 3. Standard Cleaning Logic
        df["date"] = (df["date"].astype(str)
            .str.replace(pattern_days, "", regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_en, lambda m: Months.MONTHS_WORDS1[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_ru, lambda m: Months.MONTHS_RUSSIAN[m.group(0).lower()], regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_abbr, lambda m: Months.MONTHS_WORDS_ABBR[m.group(0).title()], regex=True, flags=re.IGNORECASE))

        # 4. Punctuation and Formatting
        df["date"] = (df["date"]
            .str.replace(r"[,/]", " ", regex=True) # Remove commas from "MARCH 9, 2026"
            .str.replace(r"\s+\d{1,2}\s*:\s*\d{2}.*$", "", regex=True) # Remove time
            .str.replace(r"\s+", "-", regex=True) # Convert all spaces to dashes
            .str.strip("-"))

        # 5. Year Append and Final Conversion
        df["date"] = df["date"].apply(lambda x: x + "-2026" if isinstance(x, str) and x.count("-") == 1 else x)
        
        # Note: JPost is typically Month-Day-Year. We ensure pd.to_datetime handles it.
        df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True).dt.strftime("%d-%m-%Y")
        
        return df

print(clean_date("jpost")["date"].isna().count())


length:  title    236
date     206
text     236
dtype: int64
236
